# **Air University Multan Campus**<br>

**LAB 6 Task<br>**

**Subject Name & Code:** SE425 Data Mining<br>
**Department:** Computer Science<br>
**Program:** BS Software Engineering<br>
**Semester:** 7 Fall 2026<br>
**Instructor:** Fazle Hanan<br>
**Date & Week:** 14 October 2026 & Week 6<br>
**Lab:** B216<br>

### Student Details

Save this notebook as `RegID_Name_Lab6Task.ipynb` before you submit it.

**Registration ID:** 233670

**Name:** Sidra Ghulam Hussain


Complete the 4 tasks below. Write your code under each `# TODO` line and run the cell. No expected output is given — read each task carefully and check your own results. Do not load any file other than the provided `wb_task6_indicators.csv` and `wb_task6_countries.csv`. Marking criteria are in the Lab 6 Task Handout rubric.

In [1]:
# Setup cell — run this first. Do not modify.
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

task_df = pd.read_csv("wb_task6_indicators.csv")
lending_df = pd.read_csv("wb_task6_countries.csv")
print(task_df.shape, lending_df.shape)
task_df.head()

(217, 6) (296, 3)


,country_code,country,under5_mortality,urban_pct,electricity_access_pct,fertility_rate
0,AFG,Afghanistan,56.8,25.393925,85.3,4.932
1,ALB,Albania,9.4,57.864348,100.0,1.355
2,DZA,Algeria,22.3,74.179586,100.0,2.817
3,ASM,American Samoa,NaN,81.794798,NaN,2.305
4,AND,Andorra,2.7,88.754877,100.0,1.071


### Task 1.1 — Handling missing data (25 marks)

Using `task_df`:
1. Store the number of missing values in every column as a pandas Series called `missing_counts` and print it.
2. Create `drop_df` by removing every row that has a missing value in `under5_mortality` or `electricity_access_pct`, and print its number of rows.
3. Create `filled_df`, a copy of `task_df` in which the missing values of `under5_mortality` are replaced by the mean of its known values and the missing values of `electricity_access_pct` are replaced by the median of its known values (compute the mean and the median before filling).
4. For `under5_mortality`, print the mean and the standard deviation (pandas default) of the known values and of the filled column, rounded to 3 decimal places, and print whether the mean changed (use `np.isclose`).

In [2]:
# TODO: count the missing values, build drop_df and filled_df, and compare the mean and standard deviation of under5_mortality before and after filling

# isna() marks each missing cell; sum() adds them up per column
missing_counts = task_df.isna().sum()
print("Missing values per column:")
print(missing_counts)

# dropna(subset=...) removes a row if it has a missing value in any of the named columns
drop_df = task_df.dropna(subset=["under5_mortality", "electricity_access_pct"])
print("\nRows after removing missing values:", len(drop_df))

# Compute the mean and the median BEFORE filling (pandas skips NaN automatically)
under5_mean = task_df["under5_mortality"].mean()
electricity_median = task_df["electricity_access_pct"].median()

# Work on a copy so task_df keeps its missing values for the later tasks
filled_df = task_df.copy()

# fillna(value) replaces every missing value in the column with value
filled_df["under5_mortality"] = filled_df["under5_mortality"].fillna(under5_mean)
filled_df["electricity_access_pct"] = filled_df["electricity_access_pct"].fillna(electricity_median)

# Known values only (the original column still has NaN, which pandas skips)
known_mean = task_df["under5_mortality"].mean()
known_std = task_df["under5_mortality"].std()

# Filled column (pandas default std divides by n - 1)
filled_mean = filled_df["under5_mortality"].mean()
filled_std = filled_df["under5_mortality"].std()

print("\nunder5_mortality                 mean      std")
print(f"Known values only:          {known_mean:9.3f} {known_std:8.3f}")
print(f"After mean substitution:    {filled_mean:9.3f} {filled_std:8.3f}")

# np.isclose avoids a false 'changed' caused by tiny floating-point errors
mean_changed = not np.isclose(known_mean, filled_mean)
print("Did the mean change?", mean_changed)


Missing values per column:
country_code               0
country                    0
under5_mortality          21
urban_pct                  0
electricity_access_pct     2
fertility_rate             0
dtype: int64

Rows after removing missing values: 195

under5_mortality                 mean      std
Known values only:             27.215   38.866
After mean substitution:       27.215   36.928
Did the mean change? False


### Task 2.1 — Noisy data: binning and smoothing (20 marks)

Using the known values of `under5_mortality` in `task_df`:
1. Sort the known values in ascending order (reset the index) and keep the 12 values at index positions 160 to 171 (inclusive) of the sorted values as `values_12`.
2. Partition `values_12` into 3 equal-frequency bins of 4 values each with `np.array_split`.
3. Smooth every bin by its mean (round the means to 3 decimal places) and, separately, by its boundaries (every value moves to the nearer of the bin's minimum and maximum).
4. Print a pandas DataFrame with one row per value and the columns `bin` (1 to 3), `original`, `bin_mean` and `bin_boundary`.
5. Print the number of distinct values in the `bin_mean` column and in the `bin_boundary` column.

In [3]:
# TODO: build values_12, split it into 3 equal-frequency bins, smooth by bin means and by bin boundaries, and print the table and the distinct-value counts

# dropna() keeps the known values only; reset_index(drop=True) gives positions 0, 1, 2, ...
known_sorted = task_df["under5_mortality"].dropna().sort_values().reset_index(drop=True)

# iloc[160:172] is positions 160 to 171 inclusive (the end of a slice is excluded)
values_12 = known_sorted.iloc[160:172].reset_index(drop=True)

mortality_bins = np.array_split(values_12, 3)

by_mean = []
for b in mortality_bins:
    # every value in the bin is replaced by the bin mean (rounded to 3 decimals)
    by_mean.extend([round(b.mean(), 3)] * len(b))

by_boundary = []
for b in mortality_bins:
    minimum = b.min()
    maximum = b.max()
    for value in b:
        # each value moves to the nearer boundary (a tie goes to the minimum)
        if abs(value - minimum) <= abs(value - maximum):
            by_boundary.append(minimum)
        else:
            by_boundary.append(maximum)

smoothing_table = pd.DataFrame({
    "bin": np.repeat([1, 2, 3], 4),
    "original": np.concatenate(mortality_bins),
    "bin_mean": by_mean,
    "bin_boundary": by_boundary
})
print(smoothing_table)

print("\nDistinct values in bin_mean:    ", smoothing_table["bin_mean"].nunique())
print("Distinct values in bin_boundary:", smoothing_table["bin_boundary"].nunique())


    bin  original  bin_mean  bin_boundary
0     1      47.3    49.225          47.3
1     1      49.5    49.225          50.1
2     1      50.0    49.225          50.1
3     1      50.1    49.225          50.1
4     2      50.3    50.975          50.3
5     2      50.7    50.975          50.3
6     2      51.0    50.975          50.3
7     2      51.9    50.975          51.9
8     3      52.1    55.250          52.1
9     3      55.9    55.250          56.8
10    3      56.2    55.250          56.8
11    3      56.8    55.250          56.8

Distinct values in bin_mean:     3
Distinct values in bin_boundary: 6


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


### Task 3.1 — Data integration: merging two tables (25 marks)

`task_df` is keyed by `country_code` and `lending_df` is keyed by `iso3`.
1. Print whether `country_code` is unique in `task_df` and whether `iso3` is unique in `lending_df`.
2. Merge the two tables with an outer join on the two key columns and `indicator=True`, store the result as `outer_df`, and print the number of rows of each kind in the `_merge` column.
3. Create `merged_df` with an inner join of `task_df` and `lending_df` on the two key columns, checked with `validate="one_to_one"`, and print its shape.
4. For `merged_df`, print the number of economies in each `lending_type` (sorted by `lending_type`).
5. Print the number of economies in `merged_df` whose `capital_city` is missing.

In [4]:
# TODO: check the keys, build outer_df with indicator=True and merged_df (inner, validated), then report the _merge counts, the shape, the lending_type counts and the missing capital cities

print("country_code unique in task_df:  ", task_df["country_code"].is_unique)
print("iso3 unique in lending_df:       ", lending_df["iso3"].is_unique)

# left_on / right_on name the key of each table; indicator=True adds the _merge column
outer_df = task_df.merge(
    lending_df,
    left_on="country_code",
    right_on="iso3",
    how="outer",
    indicator=True
)
print("\nRows of each kind in _merge:")
print(outer_df["_merge"].value_counts())

# validate="one_to_one" raises an error if a key is repeated in either table
merged_df = task_df.merge(
    lending_df,
    left_on="country_code",
    right_on="iso3",
    how="inner",
    validate="one_to_one"
)
print("\nMerged table shape:", merged_df.shape)

print("\nEconomies per lending_type:")
print(merged_df["lending_type"].value_counts().sort_index())

print("\nEconomies with a missing capital_city:", merged_df["capital_city"].isna().sum())


country_code unique in task_df:   True
iso3 unique in lending_df:        True

Rows of each kind in _merge:
_merge
both          217
right_only     79
left_only       0
Name: count, dtype: int64

Merged table shape: (217, 9)

Economies per lending_type:
lending_type
Blend             19
IBRD              67
IDA               59
Not classified    72
Name: count, dtype: int64

Economies with a missing capital_city: 6


### Task 4.1 — Data transformation: normalization, standardization and log (30 marks)

1. For the column `urban_pct` of `task_df` (it has no missing values), compute the min-max normalized values `urban_min_max` and the z-scores `urban_z` directly from the formulas (use the population standard deviation, `ddof=0`, for the z-scores).
2. Confirm both results against scikit-learn's `MinMaxScaler` and `StandardScaler` and print the largest absolute difference for each.
3. Print the mean and the population standard deviation of `urban_z`, rounded to 6 decimal places.
4. Using the known values of `under5_mortality`, print the skewness before the transformation, after `np.log` and after `np.log1p`, rounded to 3 decimal places.
5. For `under5_mortality` (known values), print the percentage of economies whose min-max normalized value is below 0.1, without the log transformation and with `np.log` applied first.

In [5]:
# TODO: build urban_min_max and urban_z by formula, compare with scikit-learn, and report the skewness and the min-max shares for under5_mortality

# Take the urban population percentage as a series
urban = task_df["urban_pct"]

# Min-max normalization: (v - min) / (max - min)
urban_min_max = (urban - urban.min()) / (urban.max() - urban.min())

# Z-score: (v - mean) / population standard deviation (ddof=0 divides by n)
urban_z = (urban - urban.mean()) / urban.std(ddof=0)

# scikit-learn expects a 2-D input, so to_frame() passes the column as a one-column table
sk_min_max = MinMaxScaler().fit_transform(urban.to_frame())[:, 0]
sk_z = StandardScaler().fit_transform(urban.to_frame())[:, 0]

# Largest absolute difference between the manual and the scikit-learn results (should be zero or tiny)
print("Largest |manual - sklearn| min-max:", np.abs(urban_min_max.to_numpy() - sk_min_max).max())
print("Largest |manual - sklearn| z-score:", np.abs(urban_z.to_numpy() - sk_z).max())

Largest |manual - sklearn| min-max: 1.1102230246251565e-16
Largest |manual - sklearn| z-score: 4.440892098500626e-16
